# PATHQ — Week 4: Quantum VQC Encoder

**Goal:** Build and test the VQC encoder. Understand every part before Week 5 training.

**By end of this notebook:**
- 3-qubit VQC built and drawing correctly in PennyLane
- Amplitude encoding verified on real patch features
- Gradient flow confirmed (parameter-shift rule working)
- VQC wrapped as PyTorch nn.Module via TorchLayer
- Visual proof: tumour patches vs normal patches in quantum output space
- IBM Quantum account tested (optional but recommended)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
import pennylane as qml
from pathlib import Path
from datasets import load_dataset
from torchvision import transforms
import timm, warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(42); np.random.seed(42)

ROOT    = Path('..')
OUT_DIR = ROOT / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

print(f'PennyLane version: {qml.__version__}')
print(f'Device: {DEVICE}')
print()

# List available PennyLane devices
print('Available quantum devices:')
for d in ['default.qubit', 'default.mixed', 'lightning.qubit']:
    try:
        dev = qml.device(d, wires=3)
        print(f'  [OK] {d}')
    except Exception as e:
        print(f'  [--] {d}: {e}')

## Cell 2 — Quantum Basics: Draw a simple circuit first

In [ ]:
# ── STEP 1: Understand what a quantum circuit looks like ──────────────────

dev_draw = qml.device('default.qubit', wires=3)

@qml.qnode(dev_draw)
def simple_vqc(x, weights):
    """
    Minimal 3-qubit VQC.
    x:       8-dim input (2^3 amplitudes)
    weights: (n_layers, 2, n_qubits) rotation angles
    """
    # Encode input as quantum state amplitudes
    qml.AmplitudeEmbedding(x, wires=[0,1,2], normalize=True)

    # Layer 1: rotations + entanglement
    qml.RY(weights[0,0,0], wires=0); qml.RY(weights[0,0,1], wires=1); qml.RY(weights[0,0,2], wires=2)
    qml.RZ(weights[0,1,0], wires=0); qml.RZ(weights[0,1,1], wires=1); qml.RZ(weights[0,1,2], wires=2)
    qml.CNOT(wires=[0,1]); qml.CNOT(wires=[1,2]); qml.CNOT(wires=[2,0])

    # Layer 2: more rotations
    qml.RY(weights[1,0,0], wires=0); qml.RY(weights[1,0,1], wires=1); qml.RY(weights[1,0,2], wires=2)
    qml.RZ(weights[1,1,0], wires=0); qml.RZ(weights[1,1,1], wires=1); qml.RZ(weights[1,1,2], wires=2)
    qml.CNOT(wires=[0,1]); qml.CNOT(wires=[1,2])

    # Measure: Pauli-Z expectation on all qubits
    return [qml.expval(qml.PauliZ(q)) for q in range(3)]

# Draw the circuit
x_test = np.random.randn(8); x_test /= np.linalg.norm(x_test)
w_test = np.random.randn(2, 2, 3) * 0.1

print('PATHQ VQC Circuit Diagram:')
print('='*60)
print(qml.draw(simple_vqc)(x_test, w_test))
print('='*60)

# Run and show output
output = simple_vqc(x_test, w_test)
print(f'\nOutput (3 Pauli-Z expectation values):')
for i, v in enumerate(output):
    print(f'  Qubit {i}: <Z> = {float(v):.4f}  '
          f'(+1 = |0>, -1 = |1>, 0 = superposition)')

## Cell 3 — Amplitude Encoding Explained

In [ ]:
# ── WHY amplitude encoding matters for PATHQ ─────────────────────────────
# A 3-qubit state can represent 2^3 = 8 values simultaneously.
# We compress the 512-dim ResNet feature to 8-dim first,
# then encode those 8 values as quantum amplitudes.

dev = qml.device('default.qubit', wires=3)

@qml.qnode(dev)
def amplitude_encoding_demo(x):
    """Show what amplitude encoding does to the quantum state."""
    qml.AmplitudeEmbedding(x, wires=[0,1,2], normalize=True)
    return qml.state()  # return full 8-amplitude quantum state

# Two different inputs: one tumour-like, one normal-like
tumour_feat  = np.array([0.8, 0.6, 0.2, 0.1, 0.9, 0.3, 0.1, 0.4])
normal_feat  = np.array([0.1, 0.2, 0.3, 0.8, 0.1, 0.7, 0.6, 0.2])

state_tumour = amplitude_encoding_demo(tumour_feat)
state_normal = amplitude_encoding_demo(normal_feat)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
basis_labels = [f'|{i:03b}>' for i in range(8)]  # |000>, |001>, ..., |111>

probs_t = np.abs(np.array(state_tumour))**2
probs_n = np.abs(np.array(state_normal))**2

axes[0].bar(basis_labels, probs_t, color='#B83A12', alpha=0.8)
axes[0].set_title('Tumour feature — quantum state', fontsize=11)
axes[0].set_xlabel('Basis state'); axes[0].set_ylabel('Probability |amplitude|²')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(basis_labels, probs_n, color='#0A6B6B', alpha=0.8)
axes[1].set_title('Normal feature — quantum state', fontsize=11)
axes[1].set_xlabel('Basis state'); axes[1].set_ylabel('Probability |amplitude|²')
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Amplitude Encoding: 8-dim features → 3-qubit quantum state', fontsize=12)
plt.tight_layout()
plt.savefig(str(OUT_DIR/'week4_amplitude_encoding.png'), dpi=120, bbox_inches='tight')
plt.show()

print('Key insight: Different features produce completely different')
print('probability distributions across basis states.')
print('The VQC learns to exploit these differences for classification.')

## Cell 4 — Build VQCEncoder as nn.Module

In [ ]:
class VQCEncoder(nn.Module):
    """
    Quantum patch feature encoder.

    Pipeline:
        (B, 512) ResNet features
            -> Linear(512, 8) projection  [classical]
            -> Tanh activation             [bound values for normalisation]
            -> L2 normalise               [required: amplitude encoding needs ||x||=1]
            -> AmplitudeEmbedding         [quantum: 8 values -> 3-qubit state]
            -> n_layers of RY, RZ, CNOT  [quantum: trainable rotations + entanglement]
            -> 3 Pauli-Z measurements     [quantum -> classical: 3-dim output]
        -> concat(projected_8, quantum_3) -> (B, 11) hybrid features

    The 11-dim output replaces the 512-dim input to the GNN.
    Smaller but richer: quantum correlations that ResNet can't capture.
    """
    def __init__(self, in_dim=512, n_qubits=3, n_layers=2,
                 device_name='default.qubit'):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.vqc_dim  = 2 ** n_qubits   # 8 for 3 qubits

        # Classical projection: 512 -> 8
        self.proj = nn.Sequential(
            nn.Linear(in_dim, self.vqc_dim),
            nn.Tanh(),  # Tanh keeps values in [-1,1], needed before normalisation
        )

        # Create quantum device
        self.dev = qml.device(device_name, wires=n_qubits)

        # Define the quantum circuit
        @qml.qnode(self.dev, interface='torch', diff_method='parameter-shift')
        def circuit(inputs, weights):
            """
            inputs:  (8,) normalised feature vector
            weights: (n_layers, 2, n_qubits) rotation angles
            Returns: list of n_qubits expectation values
            """
            # Amplitude encoding: map 8 classical values to quantum amplitudes
            qml.AmplitudeEmbedding(inputs, wires=range(n_qubits), normalize=True)

            # Variational layers
            for l in range(n_layers):
                # RY rotations (Y-axis Bloch sphere)
                for q in range(n_qubits):
                    qml.RY(weights[l, 0, q], wires=q)
                # RZ rotations (Z-axis Bloch sphere)
                for q in range(n_qubits):
                    qml.RZ(weights[l, 1, q], wires=q)
                # CNOT entanglement: create quantum correlations between qubits
                for q in range(n_qubits - 1):
                    qml.CNOT(wires=[q, q+1])
                qml.CNOT(wires=[n_qubits-1, 0])  # wrap-around

            return [qml.expval(qml.PauliZ(q)) for q in range(n_qubits)]

        # Wrap quantum circuit as PyTorch layer
        # This enables joint training with the GNN
        weight_shapes = {'weights': (n_layers, 2, n_qubits)}
        self.vqc = qml.qnn.TorchLayer(circuit, weight_shapes)

        # Output: 8 (projected) + n_qubits (quantum measurements) = 11
        self.out_dim = self.vqc_dim + n_qubits

    def forward(self, x):
        """
        x: (B, 512)
        Returns: (B, 11) hybrid classical-quantum features
        """
        # Step 1: Project 512 -> 8
        projected = self.proj(x)                                 # (B, 8)

        # Step 2: L2-normalise (REQUIRED by AmplitudeEmbedding)
        projected_norm = F.normalize(projected, p=2, dim=1)      # (B, 8)

        # Step 3: Run VQC on each sample
        quantum_out = self.vqc(projected_norm)                   # (B, 3)

        # Step 4: Concatenate classical + quantum
        return torch.cat([projected, quantum_out], dim=1)        # (B, 11)


# ── Test the VQCEncoder ────────────────────────────────────────────────────
print('Building VQCEncoder (3 qubits, 2 layers)...')
vqc = VQCEncoder(in_dim=512, n_qubits=3, n_layers=2).to(DEVICE)

n_params = sum(p.numel() for p in vqc.parameters() if p.requires_grad)
print(f'VQCEncoder trainable parameters: {n_params}')
print(f'  Projection weights: {512*8 + 8}')   # Linear + bias
print(f'  VQC rotation angles: {2*2*3}')       # n_layers * 2 * n_qubits
print(f'Output dimension: {vqc.out_dim}  (8 classical + 3 quantum)')

# Test forward pass
test_batch = torch.randn(4, 512).to(DEVICE)
with torch.no_grad():
    out = vqc(test_batch)
print(f'\nForward pass: input {test_batch.shape} -> output {out.shape}')
print(f'Output range: [{out.min():.3f}, {out.max():.3f}]')
print('\nVQCEncoder working correctly.')

## Cell 5 — Verify Gradient Flow (Critical for training)

In [ ]:
print('Verifying gradient flow through VQC...')
print('(This uses the parameter-shift rule — quantum backpropagation)')
print()

vqc_test = VQCEncoder(in_dim=512, n_qubits=3, n_layers=2)
x_in  = torch.randn(2, 512, requires_grad=False)
y_tgt = torch.tensor([1.0, 0.0])

# Forward pass
out = vqc_test(x_in)               # (2, 11)
pred = out[:, -1]                  # use last quantum measurement as prediction
loss = F.mse_loss(pred, y_tgt)

# Backward pass
loss.backward()

print('Gradient check:')
all_ok = True
for name, param in vqc_test.named_parameters():
    has_grad = param.grad is not None and not torch.isnan(param.grad).any()
    status = 'OK' if has_grad else 'FAIL'
    if not has_grad: all_ok = False
    grad_norm = param.grad.norm().item() if has_grad else 0.0
    print(f'  [{status}] {name:30s}  shape={list(param.shape)}  '
          f'grad_norm={grad_norm:.4f}')

print()
if all_ok:
    print('ALL GRADIENTS FLOWING CORRECTLY')
    print('The parameter-shift rule is working.')
    print('VQC is ready for joint training with the GNN.')
else:
    print('WARNING: Some gradients are None or NaN')
    print('Check PennyLane version compatibility.')

## Cell 6 — VQC on Real ResNet Features (PatchCamelyon)

In [ ]:
# Extract real ResNet features from PatchCamelyon patches
# and run them through the VQC to visualise the quantum output space

print('Extracting features from PatchCamelyon...')
pcam = load_dataset('1aurent/PatchCamelyon')

backbone  = timm.create_model('resnet50', pretrained=True, num_classes=0)
extractor = backbone.to(DEVICE).eval()
for p in extractor.parameters(): p.requires_grad = False

tfm = transforms.Compose([
    transforms.Resize((96,96)), transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Get 50 tumour + 50 normal patches
n_each = 50
tumour_imgs = [tfm(pcam['train'][i]['image'])
               for i in range(len(pcam['train'])) if pcam['train'][i]['label']==1][:n_each]
normal_imgs = [tfm(pcam['train'][i]['image'])
               for i in range(len(pcam['train'])) if pcam['train'][i]['label']==0][:n_each]

tumour_batch = torch.stack(tumour_imgs).to(DEVICE)
normal_batch = torch.stack(normal_imgs).to(DEVICE)

with torch.no_grad():
    tumour_feats = extractor(tumour_batch).cpu()   # (50, 512)
    normal_feats = extractor(normal_batch).cpu()   # (50, 512)

print(f'ResNet features: tumour {tumour_feats.shape}, normal {normal_feats.shape}')

# Run through VQC
print('Running through VQC...')
vqc_fresh = VQCEncoder(in_dim=512, n_qubits=3, n_layers=2)
vqc_fresh.eval()

with torch.no_grad():
    tum_out = vqc_fresh(tumour_feats)   # (50, 11)
    nor_out = vqc_fresh(normal_feats)   # (50, 11)

# Extract quantum outputs (last 3 dims = Pauli-Z measurements)
tum_q = tum_out[:, -3:].numpy()   # (50, 3)  qubit measurements
nor_q = nor_out[:, -3:].numpy()   # (50, 3)

# Visualise quantum output distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('VQC Output: Tumour vs Normal (3 Qubit Measurements)', fontsize=12)

for q in range(3):
    axes[q].hist(tum_q[:,q], bins=20, alpha=0.7, color='#B83A12', label='Tumour', density=True)
    axes[q].hist(nor_q[:,q], bins=20, alpha=0.7, color='#0A6B6B', label='Normal', density=True)
    axes[q].set_title(f'Qubit {q} — <Z> distribution')
    axes[q].set_xlabel('<Z> expectation value  (-1 to +1)')
    axes[q].set_ylabel('Density')
    axes[q].legend()
    axes[q].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(str(OUT_DIR/'week4_vqc_distributions.png'), dpi=120, bbox_inches='tight')
plt.show()

# Statistical test: are the distributions different?
from scipy.stats import ks_2samp
print('\nKolmogorov-Smirnov test (are distributions different?)')
print('p < 0.05 means quantum output distinguishes tumour from normal')
for q in range(3):
    stat, p = ks_2samp(tum_q[:,q], nor_q[:,q])
    sig = 'SIGNIFICANT' if p < 0.05 else 'not significant'
    print(f'  Qubit {q}: KS={stat:.3f}  p={p:.4f}  [{sig}]')

## Cell 7 — 3D Scatter: quantum space visualisation

In [ ]:
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure(figsize=(10, 8))
ax  = fig.add_subplot(111, projection='3d')

ax.scatter(tum_q[:,0], tum_q[:,1], tum_q[:,2],
           c='#B83A12', alpha=0.7, s=30, label='Tumour')
ax.scatter(nor_q[:,0], nor_q[:,1], nor_q[:,2],
           c='#0A6B6B', alpha=0.7, s=30, label='Normal')

ax.set_xlabel('Qubit 0 <Z>'); ax.set_ylabel('Qubit 1 <Z>'); ax.set_zlabel('Qubit 2 <Z>')
ax.set_title('3D Quantum Feature Space\n(3 qubit measurements per patch)', fontsize=11)
ax.legend()

plt.tight_layout()
plt.savefig(str(OUT_DIR/'week4_quantum_3d_space.png'), dpi=120, bbox_inches='tight')
plt.show()

print('Key insight: If the clusters separate after training,')
print('the VQC has learned to distinguish tumour from normal tissue.')
print('With random weights (untrained), clusters overlap — that is expected.')
print('Training the VQC (Week 5) will push the clusters apart.')

## Cell 8 — Bloch Sphere Trajectory (XAI Layer 3 preview)

In [ ]:
# Preview of Week 6 XAI: the Bloch sphere shows HOW the quantum state
# evolves through the VQC layers — different for tumour vs normal.

dev_state = qml.device('default.qubit', wires=3)

# Get VQC weights from our encoder
weights_np = list(vqc_fresh.vqc.parameters())[0].detach().numpy()  # (2, 2, 3)

def get_state_at_layer(x_input, weights, after_layer):
    """Returns full quantum state vector after 'after_layer' VQC layers."""
    @qml.qnode(dev_state)
    def circuit():
        qml.AmplitudeEmbedding(x_input, wires=range(3), normalize=True)
        for l in range(after_layer):
            for q in range(3): qml.RY(weights[l,0,q], wires=q)
            for q in range(3): qml.RZ(weights[l,1,q], wires=q)
            for q in range(2): qml.CNOT(wires=[q,q+1])
            qml.CNOT(wires=[2,0])
        return qml.state()
    return np.array(circuit())


def state_to_bloch(state, qubit=0):
    """Extract Bloch sphere (theta, phi) for a single qubit from state vector."""
    state_2d = state.reshape([2]*3)
    # Partial trace: probability of |0> and |1> for qubit
    p0 = np.sum(np.abs(state_2d.take(0, axis=qubit))**2)
    p1 = np.sum(np.abs(state_2d.take(1, axis=qubit))**2)
    theta = 2 * np.arccos(np.sqrt(np.clip(p0, 0, 1)))
    return theta, 0.0  # phi simplified


# Get projected, normalised inputs for 1 tumour and 1 normal patch
with torch.no_grad():
    tum_proj  = F.normalize(vqc_fresh.proj(tumour_feats[:1]), p=2, dim=1).numpy()[0]
    norm_proj = F.normalize(vqc_fresh.proj(normal_feats[:1]), p=2, dim=1).numpy()[0]

# Track Bloch angles at each layer for qubit 0
n_layers = 2
tum_bloch  = [state_to_bloch(get_state_at_layer(tum_proj,  weights_np, l), qubit=0)
              for l in range(n_layers+1)]
norm_bloch = [state_to_bloch(get_state_at_layer(norm_proj, weights_np, l), qubit=0)
              for l in range(n_layers+1)]

print('Bloch sphere theta (polar angle) at each circuit layer:')
print(f'  Layer: Input   Layer 1   Layer 2')
print(f'  Tumour: {tum_bloch[0][0]:.3f}   {tum_bloch[1][0]:.3f}    {tum_bloch[2][0]:.3f}')
print(f'  Normal: {norm_bloch[0][0]:.3f}   {norm_bloch[1][0]:.3f}    {norm_bloch[2][0]:.3f}')

# Simple trajectory plot
fig, ax = plt.subplots(figsize=(8, 4))
layers = range(n_layers+1)
ax.plot(layers, [b[0] for b in tum_bloch],  'o-', color='#B83A12',
        lw=2, ms=8, label='Tumour')
ax.plot(layers, [b[0] for b in norm_bloch], 's-', color='#0A6B6B',
        lw=2, ms=8, label='Normal')
ax.axhline(np.pi/2, color='gray', ls='--', alpha=0.5, label='Superposition (theta=pi/2)')
ax.set_xticks(layers)
ax.set_xticklabels(['After\nEncoding', 'After\nLayer 1', 'After\nLayer 2'])
ax.set_ylabel('Bloch sphere theta (radians)')
ax.set_title('Qubit 0 — State trajectory through VQC layers\n(Quantum XAI Layer 3 preview)', fontsize=11)
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(str(OUT_DIR/'week4_bloch_trajectory.png'), dpi=120, bbox_inches='tight')
plt.show()

print('\nThis trajectory diverges MORE after training.')
print('That divergence is the quantum XAI signal.')

## Cell 9 — IBM Quantum test (optional)

In [ ]:
# OPTIONAL: Run one small circuit on real IBM Quantum hardware.
# This is not needed for training — we use the CPU simulator.
# But it's useful for the paper: 'validated on real quantum hardware'.

# Register free at: https://quantum.ibm.com
# Then copy your API token from: quantum.ibm.com/account

IBM_TOKEN = None  # paste your token here: IBM_TOKEN = 'your_token_here'

if IBM_TOKEN:
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService
        QiskitRuntimeService.save_account(channel='ibm_quantum', token=IBM_TOKEN, overwrite=True)
        service = QiskitRuntimeService(channel='ibm_quantum')
        backends = service.backends()
        print(f'IBM Quantum connected. Available devices:')
        for b in backends[:5]:
            print(f'  {b.name}  ({b.num_qubits} qubits)')
        print()
        print('Use device_name="qiskit.ibm" in VQCEncoder for real hardware runs.')
        print('Use sparingly — free tier has limited runtime per month.')
    except Exception as e:
        print(f'IBM connection failed: {e}')
        print('Continue with qiskit-aer simulator — that is fine for training.')
else:
    print('IBM_TOKEN not set — using local CPU simulator for all training.')
    print('To test real hardware: register at quantum.ibm.com and set IBM_TOKEN above.')
    print('The CPU simulator is functionally identical for training purposes.')

In [ ]:
print('WEEK 4 COMPLETE')
print('='*50)
print('What you built:')
print('  VQCEncoder (3 qubits, 2 layers, 11-dim output)')
print('  Gradient flow verified (parameter-shift rule)')
print('  Quantum output distributions visualised')
print('  Bloch sphere trajectory preview')
print()
print('Saved:')
print('  outputs/week4_amplitude_encoding.png')
print('  outputs/week4_vqc_distributions.png')
print('  outputs/week4_quantum_3d_space.png')
print('  outputs/week4_bloch_trajectory.png')
print()
print('Next: week5_hybrid_training.ipynb')
print('You will plug VQCEncoder into ClassicalGNN and compare AUC.')